# Financial Time Series and Macroeconomic Analysis

## 1. Research Objective

The objective of this notebook is to investigate the statistical properties of financial time series and the economic relationships between financial markets, interest rates and macroeconomic variables.

The research focuses on National Grid (NG.L) as the pilot asset, while retaining a reusable multi-asset methodology.

The analysis will investigate:

- Financial return distributions and statistical properties.
- Stationarity and non-stationarity.
- Serial dependence and autocorrelation.
- Volatility dynamics and clustering.
- Interest rate dynamics and yield curve behaviour.
- Macroeconomic and cross-asset relationships.

## 2. Research Scope

### Financial Markets

- National Grid (NG.L) — Pilot asset.
- SSE plc (SSE.L) — Sector peer.
- Severn Trent (SVT.L) — Sector peer.
- FTSE 100 (^FTSE) — Market benchmark.

### Interest Rates

- Bank of England Bank Rate.
- Sterling Overnight Index Average (SONIA).
- UK Gilt 2-Year Yield.
- UK Gilt 10-Year Yield.

### Macroeconomic Indicators

- UK Consumer Price Index (CPI).
- UK Core CPI.
- UK Consumer Prices Index including owner occupiers' housing costs (CPIH).
- UK Real Gross Domestic Product (GDP).
- UK GDP Growth.

## 3. Research Principles

- Preserve the original Raw datasets.
- Perform exploratory transformations only on derived working data.
- Respect source-specific frequencies and observation dates.
- Do not fabricate historical publication timestamps.
- Do not assume point-in-time validity where it has not been established.
- Do not interpret correlation as causation.
- Do not introduce predictive features or trading signals at this stage.
- Ensure that all statistical procedures are reproducible.

## 4. Data Limitations

Previously documented data quality and point-in-time limitations must be preserved throughout this analysis.

In particular:

- Equity corporate-action information is incomplete.
- GBP and GBp denomination requires explicit handling.
- SONIA methodology changed on 23 April 2018.
- Gilt historical availability and vintages remain unvalidated.
- GDP historical vintages and publication timestamps are unavailable.
- Macroeconomic publication delays and revisions must be respected.

## 5. Research Questions

The analysis will address the following questions:

1. What statistical properties characterise the financial return series?
2. Which series exhibit evidence of non-stationarity?
3. Is there statistically significant serial dependence?
4. Is there evidence of volatility clustering?
5. How do UK interest rates and government bond yields behave over time?
6. Are there economically meaningful relationships between NG.L and macroeconomic variables?
7. Which findings justify further investigation in subsequent research phases?

## 5. Imports and Configuration

This section imports the libraries required for statistical analysis, time series investigation and visualisation.

The research environment will use a consistent configuration to support reproducibility across experiments.

In [6]:
# 1. Standard library
import warnings
from pathlib import Path

import scipy
import statsmodels
import matplotlib

# 2. Data manipulation
import numpy as np
import pandas as pd

# 3. Statistical analysis
from scipy import stats
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch

# 4. Visualisation
import matplotlib.pyplot as plt

# 5. Reproducibility
RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)

# 6. Display configuration
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", lambda value: f"{value:,.6f}")

# 7. Visualisation configuration
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.grid"] = True
plt.rcParams["figure.dpi"] = 100

# 8. Warnings configuration
warnings.filterwarnings("default")

print("Research environment successfully configured.")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"SciPy version: {scipy.__version__}")
print(f"Statsmodels version: {statsmodels.__version__}")
print(f"Matplotlib version: {matplotlib.__version__}")

Research environment successfully configured.
NumPy version: 2.5.4
Pandas version: 3.1.0
SciPy version: 1.18.1
Statsmodels version: 0.15.0
Matplotlib version: 3.11.2


## 6. Project Paths and Data Configuration

This section defines the project directory structure and centralises the paths required to access the existing datasets.

The configuration uses relative path resolution to avoid dependencies on machine-specific directory locations.

Raw datasets will remain immutable. This notebook will only read existing source data and will not modify original files.

In [7]:
# 1. Identify the project root directory
current_path = Path.cwd()

project_root_candidates = [
    current_path,
    *current_path.parents
]

PROJECT_ROOT = next(
    (
        path for path in project_root_candidates
        if (path / "data" / "raw").is_dir()
        and (path / "notebooks").is_dir()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Project root could not be identified. "
        "Expected 'data/raw' and 'notebooks' directories."
    )


# 2. Define project directories
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"

DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"

MARKET_DATA_DIR = RAW_DATA_DIR / "market"
MACRO_DATA_DIR = RAW_DATA_DIR / "macro"

YAHOO_FINANCE_DIR = MARKET_DATA_DIR / "yahoo_finance"
BOE_DATA_DIR = MACRO_DATA_DIR / "bank_of_england"
ONS_DATA_DIR = MACRO_DATA_DIR / "ons"


# 3. Define the research universe
ASSET_UNIVERSE = ["NG.L", "SSE.L", "SVT.L"]
MARKET_BENCHMARK = "^FTSE"

PILOT_ASSET = "NG.L"

# 4. Validate essential directories
required_directories = {
    "Project root": PROJECT_ROOT,
    "Raw data": RAW_DATA_DIR,
    "Market data": MARKET_DATA_DIR,
    "Macroeconomic data": MACRO_DATA_DIR,
    "Yahoo Finance": YAHOO_FINANCE_DIR,
    "Bank of England": BOE_DATA_DIR,
    "ONS": ONS_DATA_DIR,
}

missing_directories = [
    name
    for name, path in required_directories.items()
    if not path.is_dir()
]

if missing_directories:
    raise FileNotFoundError(
        f"Missing expected directories: {missing_directories}"
    )


# 5. Display configuration
print("Project configuration successfully loaded.")
print(f"\nProject root: {PROJECT_ROOT}")
print(f"Raw data directory: {RAW_DATA_DIR}")
print(f"Market data directory: {MARKET_DATA_DIR}")
print(f"Macroeconomic data directory: {MACRO_DATA_DIR}")

print(f"\nPilot asset: {PILOT_ASSET}")
print(f"Research universe: {ASSET_UNIVERSE}")
print(f"Market benchmark: {MARKET_BENCHMARK}")

print("\nAll required directories have been validated.")

Project configuration successfully loaded.

Project root: /home/anderjcruz/projects/Quantitative-Trading-Strategy
Raw data directory: /home/anderjcruz/projects/Quantitative-Trading-Strategy/data/raw
Market data directory: /home/anderjcruz/projects/Quantitative-Trading-Strategy/data/raw/market
Macroeconomic data directory: /home/anderjcruz/projects/Quantitative-Trading-Strategy/data/raw/macro

Pilot asset: NG.L
Research universe: ['NG.L', 'SSE.L', 'SVT.L']
Market benchmark: ^FTSE

All required directories have been validated.


## 7. Data Acquisition Registry

This section identifies the existing data acquisition records and their associated metadata files.

The registry will provide a traceable reference to the datasets used throughout this research.

Each acquisition must remain independently identifiable through its acquisition identifier and source metadata.

Original Raw datasets and acquisition records must remain unchanged.

In [8]:
# 1. Discover acquisition metadata files
manifest_files = sorted(
    path
    for path in RAW_DATA_DIR.rglob("*")
    if path.is_file()
    and "manifest" in path.name.lower()
)

# 2. Display discovered files
if not manifest_files:
    print("No manifest files were discovered.")
else:
    print(f"Discovered {len(manifest_files)} manifest file(s):\n")

    for index, file_path in enumerate(manifest_files, start=1):
        relative_path = file_path.relative_to(PROJECT_ROOT)

        print(f"{index}. {relative_path}")
        print(f"   Format: {file_path.suffix or 'No extension'}")
        print(f"   Size: {file_path.stat().st_size:,} bytes")

Discovered 26 manifest file(s):

1. data/raw/macro/bank_of_england/uk_interest_rates/bank_rate/acquisition_20261103T162202Z_855395bd/manifest.json
   Format: .json
   Size: 806 bytes
2. data/raw/macro/bank_of_england/uk_interest_rates/bank_rate/acquisition_20261103T222444Z_01911a16/manifest.json
   Format: .json
   Size: 733 bytes
3. data/raw/macro/bank_of_england/uk_interest_rates/bank_rate/acquisition_20261103T222444Z_0a6ac262/manifest.json
   Format: .json
   Size: 732 bytes
4. data/raw/macro/bank_of_england/uk_interest_rates/bank_rate/acquisition_20261103T222444Z_10686ba9/manifest.json
   Format: .json
   Size: 732 bytes
5. data/raw/macro/bank_of_england/uk_interest_rates/bank_rate/acquisition_20261103T222444Z_1ce4a419/manifest.json
   Format: .json
   Size: 733 bytes
6. data/raw/macro/bank_of_england/uk_interest_rates/bank_rate/acquisition_20261103T222444Z_3fb1547a/manifest.json
   Format: .json
   Size: 733 bytes
7. data/raw/macro/bank_of_england/uk_interest_rates/bank_rate/acqui

In [9]:
# 1. Import JSON support
import json
from collections import Counter, defaultdict

# 2. Inspect the structure of all discovered manifests
manifest_schemas = defaultdict(list)

for manifest_path in manifest_files:

    with manifest_path.open("r", encoding="utf-8") as file:
        manifest_data = json.load(file)

    # Record top-level field names and their data types
    schema_signature = tuple(
        sorted(
            (key, type(value).__name__)
            for key, value in manifest_data.items()
        )
    )

    manifest_schemas[schema_signature].append(manifest_path)

# 3. Display schema variations
print("MANIFEST SCHEMA INSPECTION")
print("=" * 60)

print(f"Total manifests inspected: {len(manifest_files)}")
print(f"Distinct schema structures: {len(manifest_schemas)}")

for schema_number, (signature, paths) in enumerate(
    manifest_schemas.items(), start=1
):

    print(f"\nSchema {schema_number}")
    print("-" * 60)

    for field_name, field_type in signature:
        print(f"{field_name}: {field_type}")

    print(f"\nNumber of manifests using this schema: {len(paths)}")

# 4. Validate JSON readability
print("\nAll discovered manifests were successfully parsed as JSON.")

MANIFEST SCHEMA INSPECTION
Total manifests inspected: 26
Distinct schema structures: 6

Schema 1
------------------------------------------------------------
acquired_file: dict
acquisition_id: str
configuration_sha256: str
dataset_ids: list
manifest_version: str
provider: str
request_parameters: dict
retrieved_at_utc: str
source_endpoint: str
status: str

Number of manifests using this schema: 18

Schema 2
------------------------------------------------------------
acquired_file: dict
acquisition_id: str
acquisition_scope: str
configuration_sha256: dict
dataset_ids: list
manifest_version: str
provider: str
retrieved_at_utc: str
source_endpoint: str
status: str
transformation_performed: bool

Number of manifests using this schema: 2

Schema 3
------------------------------------------------------------
acquired_file: dict
acquisition_id: str
acquisition_scope: str
configuration_sha256: str
dataset_ids: list
manifest_version: str
provider: str
retrieved_at_utc: str
source_dataset_code:

## 8. Data Acquisition Registry

The Data Acquisition Registry provides a consolidated reference to all discovered data acquisition records.

The registry uses a canonical structure to facilitate dataset identification and traceability across different providers and manifest schemas.

Original manifest structures are preserved. Missing fields are represented as null values rather than inferred or reconstructed.

The registry is an analytical reference layer and does not modify the original acquisition records or Raw datasets.

In [10]:
# 1. Map manifest paths to their identified schema structures
manifest_schema_map = {}

for schema_number, (_, paths) in enumerate(
    manifest_schemas.items(), start=1
):
    for path in paths:
        manifest_schema_map[path] = f"Schema {schema_number}"


# 2. Define the canonical registry fields
registry_records = []

for manifest_path in manifest_files:

    with manifest_path.open("r", encoding="utf-8") as file:
        manifest_data = json.load(file)

    # Preserve the original nested file metadata
    file_metadata = manifest_data.get(
        "files",
        manifest_data.get("acquired_file")
    )

    # Extract canonical metadata without inference
    record = {
        "acquisition_id": manifest_data.get("acquisition_id"),
        "provider": manifest_data.get("provider"),
        "dataset_ids": manifest_data.get("dataset_ids"),
        "retrieved_at_utc": manifest_data.get("retrieved_at_utc"),
        "status": manifest_data.get("status"),
        "source_endpoint": manifest_data.get("source_endpoint"),
        "ticker": manifest_data.get("ticker"),
        "asset_name": manifest_data.get("asset_name"),
        "frequency": manifest_data.get("frequency"),
        "currency": manifest_data.get("currency"),
        "record_count": manifest_data.get("record_count"),
        "observation_start": manifest_data.get("observation_start"),
        "observation_end": manifest_data.get("observation_end"),
        "source_dataset_code": manifest_data.get(
            "source_dataset_code"
        ),
        "manifest_schema": manifest_schema_map[manifest_path],
        "manifest_path": str(
            manifest_path.relative_to(PROJECT_ROOT)
        ),
        "file_metadata": file_metadata,
    }

    registry_records.append(record)


# 3. Create the acquisition registry
acquisition_registry = pd.DataFrame(registry_records)


# 4. Validate registry construction
assert len(acquisition_registry) == len(manifest_files)

assert acquisition_registry["acquisition_id"].notna().all()

assert acquisition_registry["manifest_path"].is_unique

# 5. Display registry summary
print("DATA ACQUISITION REGISTRY")
print("=" * 60)

print(f"Total registered acquisitions: {len(acquisition_registry)}")

print(
    f"Unique acquisition IDs: "
    f"{acquisition_registry['acquisition_id'].nunique()}"
)

print("\nAcquisitions by provider:")
print(
    acquisition_registry["provider"]
    .value_counts(dropna=False)
    .to_string()
)

print("\nAcquisitions by manifest schema:")
print(
    acquisition_registry["manifest_schema"]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\nAcquisition status distribution:")
print(
    acquisition_registry["status"]
    .value_counts(dropna=False)
    .to_string()
)

print("\nRegistry columns:")
print(acquisition_registry.columns.tolist())

print("\nRegistry construction and structural checks completed.")

DATA ACQUISITION REGISTRY
Total registered acquisitions: 26
Unique acquisition IDs: 26

Acquisitions by provider:
provider
bank_of_england                   20
Yahoo Finance                      4
office_for_national_statistics     2

Acquisitions by manifest schema:
manifest_schema
Schema 1    18
Schema 2     2
Schema 3     2
Schema 4     1
Schema 5     2
Schema 6     1

Acquisition status distribution:
status
success                  23
success_with_warnings     3

Registry columns:
['acquisition_id', 'provider', 'dataset_ids', 'retrieved_at_utc', 'status', 'source_endpoint', 'ticker', 'asset_name', 'frequency', 'currency', 'record_count', 'observation_start', 'observation_end', 'source_dataset_code', 'manifest_schema', 'manifest_path', 'file_metadata']

Registry construction and structural checks completed.


In [11]:
# 1. Inspect file metadata structures by manifest schema

print("ACQUISITION FILE METADATA INSPECTION")
print("=" * 70)

for schema_name in sorted(
    acquisition_registry["manifest_schema"].unique()
):

    schema_records = acquisition_registry[
        acquisition_registry["manifest_schema"] == schema_name
    ]

    sample_record = schema_records.iloc[0]

    print(f"\n{schema_name}")
    print("-" * 70)

    print(
        json.dumps(
            sample_record["file_metadata"],
            indent=4,
            ensure_ascii=False,
            default=str
        )
    )

ACQUISITION FILE METADATA INSPECTION

Schema 1
----------------------------------------------------------------------
{
    "file_name": "original_data.csv",
    "format": "csv",
    "size_bytes": 231447,
    "sha256": "bbe3b2768bc510461339a0a5fc55c787c49adfc91f838ffe5b9791b104615188"
}

Schema 2
----------------------------------------------------------------------
{
    "file_name": "original_data.zip",
    "format": "zip",
    "size_bytes": 39275836,
    "sha256": "3b0c30fd38b9148731d66f7c2b1f9f57a84890b0cf021e4bd1a2c803b868211a",
    "workbook_count": 8,
    "workbooks": [
        "GLC Nominal daily data_1979 to 1984.xlsx",
        "GLC Nominal daily data_1985 to 1989.xlsx",
        "GLC Nominal daily data_1990 to 1994.xlsx",
        "GLC Nominal daily data_1995 to 1999.xlsx",
        "GLC Nominal daily data_2000 to 2004.xlsx",
        "GLC Nominal daily data_2005 to 2015.xlsx",
        "GLC Nominal daily data_2016 to 2024.xlsx",
        "GLC Nominal daily data_2025 to present.xlsx

## 9. Acquisition File Traceability

This section verifies the relationship between acquisition manifests and their associated files.

The validation checks:

- File references declared in each manifest.
- File existence within the corresponding acquisition directory.
- Actual file sizes against recorded metadata.
- Preservation of recorded SHA-256 checksums.

ZIP archives are treated as individual acquired files. Their internal contents are not registered as separate acquisitions.

Original files remain unchanged.

In [12]:
# 1. Build a file-level traceability registry

file_registry_records = []

for _, acquisition in acquisition_registry.iterrows():

    manifest_path = PROJECT_ROOT / acquisition["manifest_path"]
    acquisition_directory = manifest_path.parent

    file_metadata = acquisition["file_metadata"]

    if not isinstance(file_metadata, dict):
        raise TypeError(
            f"Invalid file metadata for acquisition "
            f"{acquisition['acquisition_id']}"
        )

    # Identify the file metadata structure
    if "file_name" in file_metadata:

        declared_files = {
            file_metadata["file_name"]: file_metadata
        }

    else:

        declared_files = file_metadata

    # Register each declared file
    for file_name, metadata in declared_files.items():

        file_path = acquisition_directory / file_name

        # Prevent references outside the acquisition directory
        resolved_path = file_path.resolve()
        resolved_directory = acquisition_directory.resolve()

        if not resolved_path.is_relative_to(resolved_directory):
            raise ValueError(
                f"File reference escapes acquisition directory: "
                f"{file_name}"
            )

        file_exists = resolved_path.is_file()

        actual_size = (
            resolved_path.stat().st_size
            if file_exists
            else None
        )

        expected_size = metadata.get("size_bytes")

        size_matches = (
            actual_size == expected_size
            if file_exists and expected_size is not None
            else None
        )

        file_registry_records.append({

            "acquisition_id": acquisition["acquisition_id"],
            "provider": acquisition["provider"],
            "manifest_schema": acquisition["manifest_schema"],
            "file_name": file_name,
            "file_format": metadata.get("format", resolved_path.suffix.lstrip(".")),
            "file_path": str(
                resolved_path.relative_to(PROJECT_ROOT)
            ),
            "file_exists": file_exists,
            "expected_size_bytes": expected_size,
            "actual_size_bytes": actual_size,
            "size_matches": size_matches,
            "expected_sha256": metadata.get("sha256"),
            "checksum_recorded": metadata.get("sha256") is not None,
        })


# 2. Create the file-level registry

file_registry = pd.DataFrame(file_registry_records)


# 3. Validate file traceability

missing_files = file_registry[
    ~file_registry["file_exists"]
]

size_mismatches = file_registry[
    file_registry["size_matches"] == False
]

missing_checksums = file_registry[
    ~file_registry["checksum_recorded"]
]


# 4. Display validation results

print("ACQUISITION FILE TRACEABILITY")
print("=" * 65)

print(f"Registered acquisitions: {len(acquisition_registry)}")
print(f"Registered files: {len(file_registry)}")

print(
    f"Existing files: "
    f"{file_registry['file_exists'].sum()}"
)

print(f"Missing files: {len(missing_files)}")

print(f"File size mismatches: {len(size_mismatches)}")

print(f"Files with recorded SHA-256: "
      f"{file_registry['checksum_recorded'].sum()}")

print(f"Files without recorded SHA-256: "
      f"{len(missing_checksums)}")


# 5. Display file formats

print("\nFiles by format:")

print(
    file_registry["file_format"]
    .value_counts(dropna=False)
    .to_string()
)


# 6. Display exceptions, if any

if not missing_files.empty:

    print("\nMISSING FILES:")
    print(
        missing_files[
            ["acquisition_id", "file_name", "file_path"]
        ].to_string(index=False)
    )

if not size_mismatches.empty:

    print("\nFILE SIZE MISMATCHES:")
    print(
        size_mismatches[
            [
                "acquisition_id",
                "file_name",
                "expected_size_bytes",
                "actual_size_bytes"
            ]
        ].to_string(index=False)
    )

if not missing_checksums.empty:

    print("\nFILES WITHOUT RECORDED CHECKSUMS:")
    print(
        missing_checksums[
            ["acquisition_id", "file_name"]
        ].to_string(index=False)
    )


print("\nFile traceability inspection completed.")

ACQUISITION FILE TRACEABILITY
Registered acquisitions: 26
Registered files: 30
Existing files: 30
Missing files: 0
File size mismatches: 0
Files with recorded SHA-256: 30
Files without recorded SHA-256: 0

Files by format:
file_format
csv        24
parquet     4
zip         2

File traceability inspection completed.


## 10. Research Dataset Selection

This section identifies the acquisition records considered for the financial time series and macroeconomic analysis.

Dataset selection follows the previously documented acquisition, data quality and temporal alignment assessments.

Acquisition identifiers are retained to ensure reproducibility and provenance.

No dataset is assumed to be point-in-time valid unless this has been established through documented evidence.

In [13]:
# 1. Display acquisition-level research inventory

inventory_columns = [
    "acquisition_id",
    "provider",
    "dataset_ids",
    "retrieved_at_utc",
    "status",
    "ticker",
    "asset_name",
    "frequency",
    "record_count",
    "observation_start",
    "observation_end",
    "manifest_schema",
]

research_inventory = acquisition_registry[
    inventory_columns
].copy()

print("RESEARCH ACQUISITION INVENTORY")
print("=" * 100)

print(
    research_inventory.to_string(
        index=False,
        max_colwidth=35
    )
)

RESEARCH ACQUISITION INVENTORY
           acquisition_id                       provider                         dataset_ids                 retrieved_at_utc                status ticker       asset_name frequency  record_count   observation_start     observation_end manifest_schema
20261103T162202Z_855395bd                bank_of_england                      [UK_BANK_RATE]      2026-11-03T16:22:02.958010Z               success    NaN              NaN       NaN           NaN                 NaN                 NaN        Schema 1
20261103T222444Z_01911a16                bank_of_england                      [UK_BANK_RATE]      2026-11-03T22:24:44.458265Z               success    NaN              NaN       NaN           NaN                 NaN                 NaN        Schema 1
20261103T222444Z_0a6ac262                bank_of_england                      [UK_BANK_RATE]      2026-11-03T22:24:44.317469Z               success    NaN              NaN       NaN           NaN                 N

In [14]:
# 1. Build a focused acquisition review table

selection_review = acquisition_registry[
    [
        "acquisition_id",
        "provider",
        "dataset_ids",
        "retrieved_at_utc",
        "status",
        "ticker",
        "record_count",
        "manifest_schema",
        "manifest_path",
        "file_metadata",
    ]
].copy()


# 2. Extract declared file names

def extract_file_names(file_metadata):

    if not isinstance(file_metadata, dict):
        return None

    if "file_name" in file_metadata:
        return file_metadata["file_name"]

    return list(file_metadata.keys())


selection_review["declared_files"] = (
    selection_review["file_metadata"]
    .apply(extract_file_names)
)


# 3. Display relevant information

display_columns = [
    "acquisition_id",
    "provider",
    "dataset_ids",
    "retrieved_at_utc",
    "status",
    "ticker",
    "record_count",
    "declared_files",
]

print("ACQUISITION SELECTION REVIEW")
print("=" * 100)

print(
    selection_review[display_columns].to_string(
        index=False,
        max_colwidth=60
    )
)

ACQUISITION SELECTION REVIEW
           acquisition_id                       provider                               dataset_ids                 retrieved_at_utc                status ticker  record_count             declared_files
20261103T162202Z_855395bd                bank_of_england                            [UK_BANK_RATE]      2026-11-03T16:22:02.958010Z               success    NaN           NaN          original_data.csv
20261103T222444Z_01911a16                bank_of_england                            [UK_BANK_RATE]      2026-11-03T22:24:44.458265Z               success    NaN           NaN          original_data.csv
20261103T222444Z_0a6ac262                bank_of_england                            [UK_BANK_RATE]      2026-11-03T22:24:44.317469Z               success    NaN           NaN          original_data.csv
20261103T222444Z_10686ba9                bank_of_england                            [UK_BANK_RATE]      2026-11-03T22:24:44.380702Z               success    NaN   

## 11. Research Dataset Configuration

This section defines the datasets selected for the current statistical research.

Dataset selection follows the documented acquisition and data quality assessments.

Each dataset retains its original acquisition identifier and associated limitations.

Datasets with unresolved acquisition or point-in-time limitations will not be silently promoted to fully validated research inputs.

In [15]:
# 1. Define the selected equity acquisitions

SELECTED_EQUITY_IDS = {
    "NG.L": "20261103T000455Z_8b505769",
    "SSE.L": "20261103T002507Z_de101edb",
    "SVT.L": "20261103T002515Z_fdac38df",
    "^FTSE": "20261103T002524Z_9c7eb120",
}

# 2. Define macroeconomic acquisition references

SELECTED_MACRO_IDS = {
    "SONIA": "20261103T224853Z_fc90361c",
    "UK_INFLATION": "20261105T132954Z_b4ce351c",
    "UK_GDP": "20261105T132956Z_60b29116",
}

# 3. Define provisional interest rate references

PROVISIONAL_INTEREST_RATE_IDS = {
    "UK_GILT_CURVE": "20261105T125522Z_eec59b41",
}

# Bank Rate remains unresolved until the previously
# documented complete acquisition ID is recovered.

BANK_RATE_ACQUISITION_ID = None

# 4. Define research limitations

DATASET_LIMITATIONS = {
    "SONIA": "Methodology change on 23 April 2018.",
    "UK_INFLATION": "Historical publication timestamps and vintages are incomplete.",
    "UK_GDP": "Historical vintages and publication timestamps are not fully validated.",
    "UK_GILT_CURVE": "Historical availability and point-in-time validity remain unvalidated.",
    "BANK_RATE": "Complete acquisition identifier must be recovered from previous validation evidence.",
}

# 5. Validate selected acquisition identifiers

all_selected_ids = (
    list(SELECTED_EQUITY_IDS.values())
    + list(SELECTED_MACRO_IDS.values())
    + list(PROVISIONAL_INTEREST_RATE_IDS.values())
)

known_ids = set(acquisition_registry["acquisition_id"])

unknown_ids = [
    acquisition_id
    for acquisition_id in all_selected_ids
    if acquisition_id not in known_ids
]

assert not unknown_ids, f"Unknown acquisition IDs: {unknown_ids}"

# 6. Display configuration

print("RESEARCH DATASET CONFIGURATION")
print("=" * 60)

print(f"Selected equity datasets: {len(SELECTED_EQUITY_IDS)}")
print(f"Selected macroeconomic datasets: {len(SELECTED_MACRO_IDS)}")
print(f"Provisional interest rate datasets: {len(PROVISIONAL_INTEREST_RATE_IDS)}")

print(f"Bank Rate acquisition: {BANK_RATE_ACQUISITION_ID}")

print("\nAll explicitly selected acquisition IDs exist in the registry.")

RESEARCH DATASET CONFIGURATION
Selected equity datasets: 4
Selected macroeconomic datasets: 3
Provisional interest rate datasets: 1
Bank Rate acquisition: None

All explicitly selected acquisition IDs exist in the registry.


In [16]:
# 1. Selected equity acquisitions

SELECTED_EQUITY_IDS = {
    "NG.L": "20261103T000455Z_8b505769",
    "SSE.L": "20261103T002507Z_de101edb",
    "SVT.L": "20261103T002515Z_fdac38df",
    "^FTSE": "20261103T002524Z_9c7eb120",
}

# 2. Selected interest rate acquisitions

SELECTED_INTEREST_RATE_IDS = {
    "BANK_RATE": "20261103T225333Z_db83b1b8",
    "SONIA": "20261103T224853Z_fc90361c",
    "UK_GILT_CURVE": "20261105T125522Z_eec59b41",
}

# 3. Equivalent complete Bank Rate acquisition

BANK_RATE_EQUIVALENT_ID = "20261103T162202Z_855395bd"

# 4. Selected macroeconomic acquisitions

SELECTED_MACRO_IDS = {
    "UK_INFLATION": "20261105T132954Z_b4ce351c",
    "UK_GDP": "20261105T132956Z_60b29116",
}

# 5. Documented limitations

DATASET_LIMITATIONS = {
    "BANK_RATE": "PIT_CONDITIONAL. Conservative availability on the next London business day.",
    "SONIA": "PIT_CONDITIONAL. Methodology change on 23 April 2018.",
    "UK_GILT_CURVE": "PIT_UNVALIDATED. Historical availability and vintages remain unvalidated.",
    "UK_INFLATION": "PIT_CONDITIONAL. Historical publication timestamps and vintages are incomplete.",
    "UK_GDP": "PIT_UNVALIDATED. Historical vintages and publication timestamps are unavailable.",
}

# 6. Validate all selected acquisition identifiers

all_selected_ids = (
    list(SELECTED_EQUITY_IDS.values())
    + list(SELECTED_INTEREST_RATE_IDS.values())
    + list(SELECTED_MACRO_IDS.values())
    + [BANK_RATE_EQUIVALENT_ID]
)

known_ids = set(acquisition_registry["acquisition_id"])

unknown_ids = [
    acquisition_id
    for acquisition_id in all_selected_ids
    if acquisition_id not in known_ids
]

assert not unknown_ids, f"Unknown acquisition IDs: {unknown_ids}"

assert (
    BANK_RATE_ACQUISITION_ID
    if "BANK_RATE_ACQUISITION_ID" in globals()
    else None
) is None or BANK_RATE_ACQUISITION_ID == SELECTED_INTEREST_RATE_IDS["BANK_RATE"]

print("FINAL RESEARCH DATASET CONFIGURATION")
print("=" * 65)

print(f"Equity datasets: {len(SELECTED_EQUITY_IDS)}")
print(f"Interest rate datasets: {len(SELECTED_INTEREST_RATE_IDS)}")
print(f"Macroeconomic datasets: {len(SELECTED_MACRO_IDS)}")

print("\nSelected Bank Rate:")
print(SELECTED_INTEREST_RATE_IDS["BANK_RATE"])

print("\nEquivalent complete Bank Rate acquisition:")
print(BANK_RATE_EQUIVALENT_ID)

print("\nAll selected acquisition IDs exist in the registry.")

FINAL RESEARCH DATASET CONFIGURATION
Equity datasets: 4
Interest rate datasets: 3
Macroeconomic datasets: 2

Selected Bank Rate:
20261103T225333Z_db83b1b8

Equivalent complete Bank Rate acquisition:
20261103T162202Z_855395bd

All selected acquisition IDs exist in the registry.


## 12. Controlled Data Loading

This section loads the selected datasets into memory for subsequent statistical analysis.

Equity datasets are loaded from their Parquet representations, while interest rate and macroeconomic datasets are loaded from their original CSV files.

The original Raw files remain unchanged.

No date conversions, imputations, resampling, forward-filling or other analytical transformations are performed during this stage.

The original acquisition identifiers are retained to ensure data provenance.

In [17]:
# 1. Initialise the working dataset dictionary

loaded_datasets = {}

dataset_acquisition_map = {}


# 2. Define a controlled file-loading function

def load_acquisition_file(acquisition_id, preferred_format):

    acquisition_files = file_registry[
        file_registry["acquisition_id"] == acquisition_id
    ]

    selected_files = acquisition_files[
        acquisition_files["file_format"].str.lower()
        == preferred_format.lower()
    ]

    if len(selected_files) != 1:
        raise ValueError(
            f"Expected exactly one {preferred_format} file for "
            f"{acquisition_id}. Found: {len(selected_files)}"
        )

    file_path = PROJECT_ROOT / selected_files.iloc[0]["file_path"]

    if not file_path.is_file():
        raise FileNotFoundError(file_path)

    if preferred_format.lower() == "parquet":
        return pd.read_parquet(file_path)

    if preferred_format.lower() == "csv":
        return pd.read_csv(file_path, low_memory=False)

    raise ValueError(
        f"Unsupported file format: {preferred_format}"
    )


# 3. Load equity datasets from Parquet

for ticker, acquisition_id in SELECTED_EQUITY_IDS.items():

    loaded_datasets[ticker] = load_acquisition_file(
        acquisition_id,
        "parquet"
    )

    dataset_acquisition_map[ticker] = acquisition_id


# 4. Load Bank of England datasets from CSV

interest_rate_datasets = {
    "BANK_RATE": SELECTED_INTEREST_RATE_IDS["BANK_RATE"],
    "SONIA": SELECTED_INTEREST_RATE_IDS["SONIA"],
}

for dataset_name, acquisition_id in interest_rate_datasets.items():

    loaded_datasets[dataset_name] = load_acquisition_file(
        acquisition_id,
        "csv"
    )

    dataset_acquisition_map[dataset_name] = acquisition_id


# 5. Load ONS datasets from CSV

for dataset_name, acquisition_id in SELECTED_MACRO_IDS.items():

    loaded_datasets[dataset_name] = load_acquisition_file(
        acquisition_id,
        "csv"
    )

    dataset_acquisition_map[dataset_name] = acquisition_id


# 6. Validate loaded datasets

assert len(loaded_datasets) == 8

assert set(loaded_datasets.keys()) == set(
    dataset_acquisition_map.keys()
)

for dataset_name, dataframe in loaded_datasets.items():

    if not isinstance(dataframe, pd.DataFrame):
        raise TypeError(
            f"{dataset_name} was not loaded as a DataFrame."
        )


# 7. Display loading summary

print("CONTROLLED DATA LOADING")
print("=" * 75)

for dataset_name, dataframe in loaded_datasets.items():

    print(f"\nDataset: {dataset_name}")
    print(f"Acquisition ID: {dataset_acquisition_map[dataset_name]}")
    print(f"Rows: {len(dataframe):,}")
    print(f"Columns: {len(dataframe.columns)}")
    print(f"Memory usage: {dataframe.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print("\nTotal datasets loaded:", len(loaded_datasets))
print("\nAll selected tabular datasets were loaded successfully.")

CONTROLLED DATA LOADING

Dataset: NG.L
Acquisition ID: 20261103T000455Z_8b505769
Rows: 7,890
Columns: 8
Memory usage: 0.54 MB

Dataset: SSE.L
Acquisition ID: 20261103T002507Z_de101edb
Rows: 9,060
Columns: 8
Memory usage: 0.62 MB

Dataset: SVT.L
Acquisition ID: 20261103T002515Z_fdac38df
Rows: 9,455
Columns: 8
Memory usage: 0.65 MB

Dataset: ^FTSE
Acquisition ID: 20261103T002524Z_9c7eb120
Rows: 11,004
Columns: 6
Memory usage: 0.59 MB

Dataset: BANK_RATE
Acquisition ID: 20261103T225333Z_db83b1b8
Rows: 13,104
Columns: 2
Memory usage: 0.34 MB

Dataset: SONIA
Acquisition ID: 20261103T224853Z_fc90361c
Rows: 7,496
Columns: 2
Memory usage: 0.19 MB

Dataset: UK_INFLATION
Acquisition ID: 20261105T132954Z_b4ce351c
Rows: 1,498
Columns: 4054
Memory usage: 51.61 MB

Dataset: UK_GDP
Acquisition ID: 20261105T132956Z_60b29116
Rows: 408
Columns: 1151
Memory usage: 4.37 MB

Total datasets loaded: 8

All selected tabular datasets were loaded successfully.


In [18]:
# 1. Define the datasets to inspect

datasets_to_inspect = loaded_datasets


# 2. Inspect dataset schemas

print("LOADED DATASET SCHEMA INSPECTION")
print("=" * 85)

for dataset_name, dataframe in datasets_to_inspect.items():

    print(f"\nDATASET: {dataset_name}")
    print("-" * 85)

    print(f"Shape: {dataframe.shape}")

    print("\nColumn data types:")
    print(
        dataframe.dtypes.value_counts().to_string()
    )

    print("\nFirst 15 column names:")

    for column in dataframe.columns[:15]:
        print(f"  - {column}")

    if len(dataframe.columns) > 15:
        print("\nLast 10 column names:")

        for column in dataframe.columns[-10:]:
            print(f"  - {column}")

    print("\nFirst 3 observations:")

    print(
        dataframe.head(3).to_string(
            index=False,
            max_cols=12
        )
    )

    print("\nMissing values:")
    print(f"Total missing cells: {dataframe.isna().sum().sum():,}")

LOADED DATASET SCHEMA INSPECTION

DATASET: NG.L
-------------------------------------------------------------------------------------
Shape: (7890, 8)

Column data types:
float64    7
int64      1

First 15 column names:
  - Adj Close
  - Close
  - Dividends
  - High
  - Low
  - Open
  - Stock Splits
  - Volume

First 3 observations:
 Adj Close      Close  Dividends       High        Low       Open  Stock Splits   Volume
267.393036 271.597504   0.000000 271.597504 271.597504 271.597504      0.000000 38163728
269.307434 273.542114   0.000000 273.542114 273.542114 273.542114      0.000000  8068642
267.393036 271.597504   0.000000 271.597504 271.597504 271.597504      0.000000 12045490

Missing values:
Total missing cells: 2

DATASET: SSE.L
-------------------------------------------------------------------------------------
Shape: (9060, 8)

Column data types:
float64    7
int64      1

First 15 column names:
  - Adj Close
  - Close
  - Dividends
  - High
  - Low
  - Open
  - Stock Split

In [19]:
# 1. Inspect temporal indexes of financial datasets

print("FINANCIAL DATASET INDEX INSPECTION")
print("=" * 75)

for dataset_name in ["NG.L", "SSE.L", "SVT.L", "^FTSE"]:

    dataframe = loaded_datasets[dataset_name]

    print(f"\nDataset: {dataset_name}")
    print(f"Index type: {type(dataframe.index).__name__}")
    print(f"Index name: {dataframe.index.name}")
    print(f"Index dtype: {dataframe.index.dtype}")

    print(f"First index values: {dataframe.index[:3].tolist()}")
    print(f"Last index values: {dataframe.index[-3:].tolist()}")

    print(f"Index unique: {dataframe.index.is_unique}")
    print(f"Index monotonic increasing: {dataframe.index.is_monotonic_increasing}")


# 2. Inspect Bank Rate and SONIA date representations

print("\n\nINTEREST RATE DATE INSPECTION")
print("=" * 75)

for dataset_name in ["BANK_RATE", "SONIA"]:

    dataframe = loaded_datasets[dataset_name]

    print(f"\nDataset: {dataset_name}")
    print(f"Date column: {dataframe.columns[0]}")
    print(f"Date dtype: {dataframe.iloc[:, 0].dtype}")

    print("First dates:")
    print(dataframe.iloc[:3, 0].tolist())

    print("Last dates:")
    print(dataframe.iloc[-3:, 0].tolist())


# 3. Inspect ONS series identifiers and metadata

print("\n\nONS DATASET STRUCTURE")
print("=" * 75)

for dataset_name in ["UK_INFLATION", "UK_GDP"]:

    dataframe = loaded_datasets[dataset_name]

    print(f"\nDataset: {dataset_name}")
    print(f"Shape: {dataframe.shape}")

    print("\nFirst 12 metadata rows:")

    print(
        dataframe.iloc[:12, :6].to_string()
    )

    print("\nLast 5 rows, first 6 columns:")

    print(
        dataframe.iloc[-5:, :6].to_string()
    )

    print("\nFirst 10 column identifiers:")
    print(dataframe.columns[:10].tolist())

FINANCIAL DATASET INDEX INSPECTION

Dataset: NG.L
Index type: DatetimeIndex
Index name: Date
Index dtype: datetime64[ms]
First index values: [Timestamp('1995-12-11 00:00:00'), Timestamp('1995-12-12 00:00:00'), Timestamp('1995-12-13 00:00:00')]
Last index values: [Timestamp('2026-10-29 00:00:00'), Timestamp('2026-10-30 00:00:00'), Timestamp('2026-11-02 00:00:00')]
Index unique: True
Index monotonic increasing: True

Dataset: SSE.L
Index type: DatetimeIndex
Index name: Date
Index dtype: datetime64[ms]
First index values: [Timestamp('1991-06-17 00:00:00'), Timestamp('1991-06-18 00:00:00'), Timestamp('1991-06-19 00:00:00')]
Last index values: [Timestamp('2026-10-29 00:00:00'), Timestamp('2026-10-30 00:00:00'), Timestamp('2026-11-02 00:00:00')]
Index unique: True
Index monotonic increasing: True

Dataset: SVT.L
Index type: DatetimeIndex
Index name: Date
Index dtype: datetime64[ms]
First index values: [Timestamp('1989-12-11 00:00:00'), Timestamp('1989-12-12 00:00:00'), Timestamp('1989-12-13 

In [20]:
# 1. Define the approved ONS research series

ONS_SERIES_SELECTION = {
    "UK_INFLATION": {
        "UK_CPI": "D7G7",
        "UK_CORE_CPI": "DKO8",
        "UK_CPIH": "L55O",
    },
    "UK_GDP": {
        "UK_REAL_GDP": "ABMI",
        "UK_GDP_GROWTH": "IHYQ",
    },
}


# 2. Extract selected series without modifying source DataFrames

ons_series_raw = {}

for dataset_name, indicators in ONS_SERIES_SELECTION.items():

    source_df = loaded_datasets[dataset_name]

    cdid_row = source_df.iloc[0].astype("string").str.strip()

    for indicator_name, cdid in indicators.items():

        matching_positions = np.flatnonzero(
            cdid_row.eq(cdid).fillna(False).to_numpy()
        )

        if len(matching_positions) != 1:
            raise ValueError(
                f"Expected one column for CDID {cdid}. "
                f"Found: {len(matching_positions)}"
            )

        column_position = matching_positions[0]

        series_df = source_df.iloc[6:, [0, column_position]].copy()

        series_df.columns = ["period_label", "value"]

        series_df["value"] = pd.to_numeric(
            series_df["value"],
            errors="coerce"
        )

        series_df = series_df.dropna(subset=["value"]).copy()

        if series_df.empty:
            raise ValueError(
                f"No numeric observations found for {cdid}"
            )

        series_df["period_label"] = (
            series_df["period_label"].astype("string").str.strip()
        )

        ons_series_raw[indicator_name] = series_df.reset_index(drop=True)


# 3. Display extraction summary

print("ONS SELECTED SERIES EXTRACTION")
print("=" * 65)

for indicator_name, dataframe in ons_series_raw.items():

    print(f"\nIndicator: {indicator_name}")
    print(f"Observations: {len(dataframe):,}")

    print("\nFirst three observations:")
    print(dataframe.head(3).to_string(index=False))

    print("\nLast three observations:")
    print(dataframe.tail(3).to_string(index=False))

print("\nFive approved ONS series extracted successfully.")

ONS SELECTED SERIES EXTRACTION

Indicator: UK_CPI
Observations: 641

First three observations:
period_label    value
        1989 5.200000
        1990 7.000000
        1991 7.500000

Last three observations:
period_label    value
    2026 JUL 2.900000
    2026 AUG 3.100000
    2026 SEP 3.500000

Indicator: UK_CORE_CPI
Observations: 641

First three observations:
period_label    value
        1989 5.100000
        1990 6.300000
        1991 7.600000

Last three observations:
period_label    value
    2026 JUL 2.600000
    2026 AUG 2.600000
    2026 SEP 2.800000

Indicator: UK_CPIH
Observations: 641

First three observations:
period_label    value
        1989 5.700000
        1990 8.000000
        1991 7.500000

Last three observations:
period_label    value
    2026 JUL 3.100000
    2026 AUG 3.300000
    2026 SEP 3.600000

Indicator: UK_REAL_GDP
Observations: 364

First three observations:
period_label          value
        1948 468,350.000000
        1949 483,721.000000
        1950

In [21]:
# Classify ONS period labels before date conversion

import re

def classify_period_label(label):

    label = str(label).strip().upper()

    if re.fullmatch(r"\d{4}", label):
        return "annual"

    if re.fullmatch(r"\d{4}\s+Q[1-4]", label):
        return "quarterly"

    if re.fullmatch(
        r"\d{4}\s+(JAN|FEB|MAR|APR|MAY|JUN|JUL|AUG|SEP|OCT|NOV|DEC)",
        label
    ):
        return "monthly"

    return "unclassified"


print("ONS PERIOD LABEL CLASSIFICATION")
print("=" * 70)

for indicator_name, dataframe in ons_series_raw.items():

    classified = dataframe["period_label"].apply(
        classify_period_label
    )

    print(f"\nIndicator: {indicator_name}")
    print("-" * 70)

    print("Period type counts:")
    print(classified.value_counts(dropna=False).to_string())

    print("\nFirst occurrence of each period type:")

    for period_type in ["annual", "quarterly", "monthly", "unclassified"]:

        positions = np.flatnonzero(
            classified.eq(period_type).to_numpy()
        )

        if len(positions) > 0:

            first_position = positions[0]
            last_position = positions[-1]

            print(
                f"{period_type}: "
                f"{dataframe.iloc[first_position]['period_label']} "
                f"-> "
                f"{dataframe.iloc[last_position]['period_label']}"
            )

    print("\nFirst and last five labels:")
    print(
        pd.concat([
            dataframe["period_label"].head(5),
            dataframe["period_label"].tail(5)
        ]).to_string(index=False)
    )

ONS PERIOD LABEL CLASSIFICATION

Indicator: UK_CPI
----------------------------------------------------------------------
Period type counts:
period_label
monthly      453
quarterly    151
annual        37

First occurrence of each period type:
annual: 1989 -> 2025
quarterly: 1989 Q1 -> 2026 Q3
monthly: 1989 JAN -> 2026 SEP

First and last five labels:
    1989
    1990
    1991
    1992
    1993
2026 MAY
2026 JUN
2026 JUL
2026 AUG
2026 SEP

Indicator: UK_CORE_CPI
----------------------------------------------------------------------
Period type counts:
period_label
monthly      453
quarterly    151
annual        37

First occurrence of each period type:
annual: 1989 -> 2025
quarterly: 1989 Q1 -> 2026 Q3
monthly: 1989 JAN -> 2026 SEP

First and last five labels:
    1989
    1990
    1991
    1992
    1993
2026 MAY
2026 JUN
2026 JUL
2026 AUG
2026 SEP

Indicator: UK_CPIH
----------------------------------------------------------------------
Period type counts:
period_label
monthly      

In [22]:
# Separate ONS observations by their original frequency

ons_series_classified = {}

for indicator_name, dataframe in ons_series_raw.items():

    classified_df = dataframe.copy()

    classified_df["frequency"] = (
        classified_df["period_label"].apply(classify_period_label)
    )

    ons_series_classified[indicator_name] = classified_df


# Create frequency-specific research datasets

ons_frequency_datasets = {}

for indicator_name, dataframe in ons_series_classified.items():

    for frequency in ["annual", "quarterly", "monthly"]:

        frequency_df = (
            dataframe.loc[
                dataframe["frequency"] == frequency,
                ["period_label", "value"]
            ]
            .copy()
            .reset_index(drop=True)
        )

        if not frequency_df.empty:

            dataset_name = f"{indicator_name}_{frequency.upper()}"

            ons_frequency_datasets[dataset_name] = frequency_df


# Display frequency separation summary

print("ONS FREQUENCY SEPARATION")
print("=" * 75)

for dataset_name, dataframe in ons_frequency_datasets.items():

    print(f"\nDataset: {dataset_name}")
    print(f"Observations: {len(dataframe):,}")

    print("First observation:")
    print(dataframe.head(1).to_string(index=False))

    print("Last observation:")
    print(dataframe.tail(1).to_string(index=False))


print("\nFrequency-specific datasets created successfully.")

ONS FREQUENCY SEPARATION

Dataset: UK_CPI_ANNUAL
Observations: 37
First observation:
period_label    value
        1989 5.200000
Last observation:
period_label    value
        2025 3.400000

Dataset: UK_CPI_QUARTERLY
Observations: 151
First observation:
period_label    value
     1989 Q1 5.000000
Last observation:
period_label    value
     2026 Q3 3.100000

Dataset: UK_CPI_MONTHLY
Observations: 453
First observation:
period_label    value
    1989 JAN 4.900000
Last observation:
period_label    value
    2026 SEP 3.500000

Dataset: UK_CORE_CPI_ANNUAL
Observations: 37
First observation:
period_label    value
        1989 5.100000
Last observation:
period_label    value
        2025 3.500000

Dataset: UK_CORE_CPI_QUARTERLY
Observations: 151
First observation:
period_label    value
     1989 Q1 5.300000
Last observation:
period_label    value
     2026 Q3 2.600000

Dataset: UK_CORE_CPI_MONTHLY
Observations: 453
First observation:
period_label    value
    1989 JAN 5.300000
Last observati

In [24]:
# Diagnose the temporal index data type

dataset_name = "UK_CPI_ANNUAL"

temporal_df = ons_temporal_datasets[dataset_name]

print("TEMPORAL INDEX DIAGNOSTICS")
print("=" * 60)

print(f"Index class: {type(temporal_df.index)}")
print(f"Index dtype: {temporal_df.index.dtype}")

print("\nFirst five index values:")
print(temporal_df.index[:5])

print("\nUnderlying element types:")
print(
    pd.Series(
        [type(value).__name__ for value in temporal_df.index]
    ).value_counts()
)

print("\nPeriod end column dtype before indexing:")

original_df = ons_frequency_datasets[dataset_name].copy()

parsed_dates = original_df["period_label"].apply(parse_ons_period)

print(parsed_dates.dtype)
print(parsed_dates.head())

TEMPORAL INDEX DIAGNOSTICS
Index class: <class 'pandas.Index'>
Index dtype: timestamp[us][pyarrow]

First five index values:
Index([1989-12-31 00:00:00, 1990-12-31 00:00:00, 1991-12-31 00:00:00, 1992-12-31 00:00:00, 1993-12-31 00:00:00],
      dtype='timestamp[us][pyarrow]', name='Period End')

Underlying element types:
Timestamp    37
Name: count, dtype: int64

Period end column dtype before indexing:
timestamp[us][pyarrow]
0    1989-12-31 00:00:00
1    1990-12-31 00:00:00
2    1991-12-31 00:00:00
3    1992-12-31 00:00:00
4    1993-12-31 00:00:00
Name: period_label, dtype: timestamp[us][pyarrow]


In [25]:
# Standardise temporal indices using native pandas datetime64

for dataset_name, dataframe in ons_temporal_datasets.items():

    dataframe.index = pd.DatetimeIndex(
        pd.to_datetime(
            dataframe.index.astype("string").to_numpy(),
            errors="raise"
        ),
        name="Period End"
    )

    if not isinstance(dataframe.index, pd.DatetimeIndex):
        raise TypeError(
            f"DatetimeIndex conversion failed: {dataset_name}"
        )


# Validate standardised indices

print("ONS STANDARDISED TEMPORAL INDICES")
print("=" * 75)

for dataset_name, dataframe in ons_temporal_datasets.items():

    print(f"\nDataset: {dataset_name}")
    print(f"Index class: {type(dataframe.index).__name__}")
    print(f"Index dtype: {dataframe.index.dtype}")
    print(f"Observations: {len(dataframe):,}")
    print(f"First period: {dataframe.index.min()}")
    print(f"Last period: {dataframe.index.max()}")
    print(f"Duplicate dates: {dataframe.index.has_duplicates}")
    print(f"Monotonic: {dataframe.index.is_monotonic_increasing}")

    assert isinstance(dataframe.index, pd.DatetimeIndex)
    assert not dataframe.index.has_duplicates
    assert dataframe.index.is_monotonic_increasing


print("\nAll ONS temporal indices standardised successfully.")

ONS STANDARDISED TEMPORAL INDICES

Dataset: UK_CPI_ANNUAL
Index class: DatetimeIndex
Index dtype: datetime64[us]
Observations: 37
First period: 1989-12-31 00:00:00
Last period: 2025-12-31 00:00:00
Duplicate dates: False
Monotonic: True

Dataset: UK_CPI_QUARTERLY
Index class: DatetimeIndex
Index dtype: datetime64[us]
Observations: 151
First period: 1989-03-31 00:00:00
Last period: 2026-09-30 00:00:00
Duplicate dates: False
Monotonic: True

Dataset: UK_CPI_MONTHLY
Index class: DatetimeIndex
Index dtype: datetime64[us]
Observations: 453
First period: 1989-01-31 00:00:00
Last period: 2026-09-30 00:00:00
Duplicate dates: False
Monotonic: True

Dataset: UK_CORE_CPI_ANNUAL
Index class: DatetimeIndex
Index dtype: datetime64[us]
Observations: 37
First period: 1989-12-31 00:00:00
Last period: 2025-12-31 00:00:00
Duplicate dates: False
Monotonic: True

Dataset: UK_CORE_CPI_QUARTERLY
Index class: DatetimeIndex
Index dtype: datetime64[us]
Observations: 151
First period: 1989-03-31 00:00:00
Last per

In [26]:
# Prepare Bank of England interest rate working datasets

interest_rate_datasets = {}

for dataset_name in ["BANK_RATE", "SONIA"]:

    source_df = loaded_datasets[dataset_name]

    working_df = source_df.copy()

    # Parse observation dates
    working_df["DATE"] = pd.to_datetime(
        working_df["DATE"],
        format="%d %b %Y",
        errors="raise"
    )

    # Convert interest rate values to numeric
    value_column = [
        column for column in working_df.columns
        if column != "DATE"
    ]

    if len(value_column) != 1:
        raise ValueError(
            f"Unexpected schema for {dataset_name}: "
            f"{working_df.columns.tolist()}"
        )

    working_df[value_column[0]] = pd.to_numeric(
        working_df[value_column[0]],
        errors="raise"
    )

    working_df = (
        working_df
        .set_index("DATE")
        .sort_index()
    )

    working_df.index = pd.DatetimeIndex(
        working_df.index,
        name="Date"
    )

    # Validate
    if working_df.index.has_duplicates:
        raise ValueError(
            f"Duplicate dates found in {dataset_name}"
        )

    if not working_df.index.is_monotonic_increasing:
        raise ValueError(
            f"Non-monotonic dates in {dataset_name}"
        )

    interest_rate_datasets[dataset_name] = working_df


# Display preparation summary

print("BANK OF ENGLAND INTEREST RATE PREPARATION")
print("=" * 75)

for dataset_name, dataframe in interest_rate_datasets.items():

    print(f"\nDataset: {dataset_name}")
    print(f"Observations: {len(dataframe):,}")
    print(f"Index class: {type(dataframe.index).__name__}")
    print(f"Index dtype: {dataframe.index.dtype}")
    print(f"First observation: {dataframe.index.min()}")
    print(f"Last observation: {dataframe.index.max()}")
    print(f"Duplicate dates: {dataframe.index.has_duplicates}")
    print(f"Monotonic: {dataframe.index.is_monotonic_increasing}")

    print("\nFirst three observations:")
    print(dataframe.head(3).to_string())

    print("\nLast three observations:")
    print(dataframe.tail(3).to_string())


print("\nInterest rate working datasets prepared successfully.")

BANK OF ENGLAND INTEREST RATE PREPARATION

Dataset: BANK_RATE
Observations: 13,104
Index class: DatetimeIndex
Index dtype: datetime64[us]
First observation: 1975-01-02 00:00:00
Last observation: 2026-11-02 00:00:00
Duplicate dates: False
Monotonic: True

First three observations:
             IUDBEDR
Date                
1975-01-02 11.500000
1975-01-03 11.500000
1975-01-06 11.500000

Last three observations:
            IUDBEDR
Date               
2026-10-29 3.750000
2026-10-30 3.750000
2026-11-02 3.750000

Dataset: SONIA
Observations: 7,496
Index class: DatetimeIndex
Index dtype: datetime64[us]
First observation: 1997-03-03 00:00:00
Last observation: 2026-10-30 00:00:00
Duplicate dates: False
Monotonic: True

First three observations:
            IUDSOIA
Date               
1997-03-03 5.860000
1997-03-04 5.820000
1997-03-05 5.890000

Last three observations:
            IUDSOIA
Date               
2026-10-28 3.733800
2026-10-29 3.735200
2026-10-30 3.734800

Interest rate working datas

In [27]:
# Locate the selected Gilt Yield Curve acquisition

gilt_acquisition_id = SELECTED_INTEREST_RATE_IDS["UK_GILT_CURVE"]

gilt_registry_entry = acquisition_registry.loc[
    acquisition_registry["acquisition_id"] == gilt_acquisition_id
].copy()

if len(gilt_registry_entry) != 1:
    raise ValueError(
        f"Expected one Gilt acquisition registry entry. "
        f"Found: {len(gilt_registry_entry)}"
    )

print("SELECTED GILT ACQUISITION")
print("=" * 65)

for column in [
    "acquisition_id",
    "provider",
    "retrieved_at_utc",
    "status",
    "source_endpoint",
    "record_count",
    "manifest_path",
    "file_metadata",
]:
    if column in gilt_registry_entry.columns:
        print(f"\n{column}:")
        print(gilt_registry_entry.iloc[0][column])

SELECTED GILT ACQUISITION

acquisition_id:
20261105T125522Z_eec59b41

provider:
bank_of_england

retrieved_at_utc:
2026-11-05T12:55:22.192228+00:00

status:
success

source_endpoint:
https://www.bankofengland.co.uk/-/media/boe/files/statistics/yield-curves/glcnominalddata.zip

record_count:
nan

manifest_path:
data/raw/macro/bank_of_england/yield_curve_archive/acquisition_20261105T125522Z_eec59b41/manifest.json

file_metadata:
{'file_name': 'original_data.zip', 'format': 'zip', 'size_bytes': 39275836, 'sha256': '3b0c30fd38b9148731d66f7c2b1f9f57a84890b0cf021e4bd1a2c803b868211a', 'workbook_count': 8, 'workbooks': ['GLC Nominal daily data_1979 to 1984.xlsx', 'GLC Nominal daily data_1985 to 1989.xlsx', 'GLC Nominal daily data_1990 to 1994.xlsx', 'GLC Nominal daily data_1995 to 1999.xlsx', 'GLC Nominal daily data_2000 to 2004.xlsx', 'GLC Nominal daily data_2005 to 2015.xlsx', 'GLC Nominal daily data_2016 to 2024.xlsx', 'GLC Nominal daily data_2025 to present.xlsx']}


In [28]:
# Extract the selected Gilt Yield Curve archive into a working directory

import zipfile
from openpyxl import load_workbook

# 1. Resolve the original archive path

gilt_manifest_path = (
    PROJECT_ROOT /
    gilt_registry_entry.iloc[0]["manifest_path"]
)

gilt_archive_path = (
    gilt_manifest_path.parent / "original_data.zip"
)

if not gilt_archive_path.exists():
    raise FileNotFoundError(
        f"Gilt archive not found: {gilt_archive_path}"
    )


# 2. Define a derived working directory

gilt_working_dir = (
    PROJECT_ROOT /
    "data" /
    "interim" /
    "macro" /
    "bank_of_england" /
    "yield_curve" /
    gilt_acquisition_id
)

gilt_working_dir.mkdir(
    parents=True,
    exist_ok=True
)


# 3. Validate archive contents before extraction

expected_workbooks = (
    gilt_registry_entry.iloc[0]["file_metadata"]["workbooks"]
)

with zipfile.ZipFile(gilt_archive_path, "r") as archive:

    archive_members = archive.namelist()

    workbook_members = [
        name for name in archive_members
        if name.lower().endswith(".xlsx")
    ]

    if set(workbook_members) != set(expected_workbooks):
        raise ValueError(
            "Archive workbook contents do not match acquisition metadata."
        )

    # Protect against unexpected directory traversal paths
    for member in archive_members:

        target_path = (gilt_working_dir / member).resolve()

        if not target_path.is_relative_to(gilt_working_dir.resolve()):
            raise ValueError(
                f"Unsafe archive member path: {member}"
            )

    # 4. Extract the archive into the derived directory
    archive.extractall(gilt_working_dir)


# 5. Inspect workbook sheets and dimensions

print("GILT YIELD CURVE ARCHIVE EXTRACTION")
print("=" * 75)

print(f"Original archive: {gilt_archive_path}")
print(f"Working directory: {gilt_working_dir}")
print(f"Expected workbooks: {len(expected_workbooks)}")

for workbook_name in expected_workbooks:

    workbook_path = gilt_working_dir / workbook_name

    if not workbook_path.exists():
        raise FileNotFoundError(workbook_path)

    workbook = load_workbook(
        workbook_path,
        read_only=True,
        data_only=True
    )

    print(f"\nWorkbook: {workbook_name}")

    for sheet_name in workbook.sheetnames:

        worksheet = workbook[sheet_name]

        print(
            f"  Sheet: {sheet_name} | "
            f"Rows: {worksheet.max_row:,} | "
            f"Columns: {worksheet.max_column:,}"
        )

    workbook.close()


print("\nGilt archive extraction and workbook inspection completed.")

GILT YIELD CURVE ARCHIVE EXTRACTION
Original archive: /home/anderjcruz/projects/Quantitative-Trading-Strategy/data/raw/macro/bank_of_england/yield_curve_archive/acquisition_20261105T125522Z_eec59b41/original_data.zip
Working directory: /home/anderjcruz/projects/Quantitative-Trading-Strategy/data/interim/macro/bank_of_england/yield_curve/20261105T125522Z_eec59b41
Expected workbooks: 8

Workbook: GLC Nominal daily data_1979 to 1984.xlsx
  Sheet: info | Rows: 14 | Columns: 13
  Sheet: 1. nominal fwds, short end | Rows: 4,442 | Columns: 61
  Sheet: 2. nominal fwd curve | Rows: 4,442 | Columns: 61
  Sheet: 3. nominal spot, short end | Rows: 4,442 | Columns: 61
  Sheet: 4. nominal spot curve | Rows: 4,442 | Columns: 61

Workbook: GLC Nominal daily data_1985 to 1989.xlsx
  Sheet: info | Rows: 14 | Columns: 13
  Sheet: 1. nominal fwds, short end | Rows: 3,870 | Columns: 81
  Sheet: 2. nominal fwd curve | Rows: 3,870 | Columns: 61
  Sheet: 3. nominal spot, short end | Rows: 3,870 | Columns: 61


In [29]:
# Inspect Gilt workbook metadata and nominal spot curve structure

from openpyxl import load_workbook

print("GILT WORKBOOK STRUCTURAL INSPECTION")
print("=" * 85)

for workbook_name in expected_workbooks:

    workbook_path = gilt_working_dir / workbook_name

    workbook = load_workbook(
        workbook_path,
        read_only=True,
        data_only=True
    )

    spot_sheet_name = next(
        (
            sheet_name
            for sheet_name in workbook.sheetnames
            if "spot curve" in sheet_name.lower()
        ),
        None
    )

    if spot_sheet_name is None:
        raise ValueError(
            f"No spot curve sheet found in {workbook_name}"
        )

    print(f"\n{'=' * 85}")
    print(f"WORKBOOK: {workbook_name}")

    # Inspect workbook metadata
    info_sheet = workbook["info"]

    print("\nINFO SHEET — FIRST 14 ROWS")

    for row in info_sheet.iter_rows(
        min_row=1,
        max_row=min(14, info_sheet.max_row),
        values_only=True
    ):
        values = [
            value for value in row
            if value is not None
        ]

        if values:
            print(values)

    # Inspect nominal spot curve structure
    spot_sheet = workbook[spot_sheet_name]

    print(f"\nSPOT CURVE SHEET: {spot_sheet_name}")
    print(
        f"Dimensions: "
        f"{spot_sheet.max_row:,} rows × "
        f"{spot_sheet.max_column:,} columns"
    )

    print("\nFIRST 12 ROWS — FIRST 20 COLUMNS")

    for row in spot_sheet.iter_rows(
        min_row=1,
        max_row=min(12, spot_sheet.max_row),
        min_col=1,
        max_col=min(20, spot_sheet.max_column),
        values_only=True
    ):
        print(row)

    workbook.close()

print("\nGilt workbook structural inspection completed.")

GILT WORKBOOK STRUCTURAL INSPECTION

WORKBOOK: GLC Nominal daily data_1979 to 1984.xlsx

INFO SHEET — FIRST 14 ROWS
['Bank of England UK yield curve data']
['For documentation and descriptions, see the statistics/yield curve area of the']
['Bank of England website at www.bankofengland.co.uk.']
['Every effort has been made to ensure this information is correct but we can not ']
['in any way guarantee its accuracy and you use it entirely at your own risk.']
['Sources: Bloomberg Finance L.P., and Bank calculations']

SPOT CURVE SHEET: 4. nominal spot curve
Dimensions: 4,442 rows × 61 columns

FIRST 12 ROWS — FIRST 20 COLUMNS
(None, 'UK nominal spot curve', None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None)
(None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None)
('Maturity', None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, 